# AI Music Creator — on a rented GPU

This repo comes in two halves, and only the first is in this clone.

| | | |
|---|---|---|
| **the tool** | `aimc/`, the wrappers, `night/*.py`, this notebook | public, and holds no song anyone wrote |
| **the content** | presets, lyrics, collections, takes | a second, private clone — step 4 |

A fresh clone of the tool alone is complete: the example presets at
`presets/` work, and you can write your own. `aimc/workspace.py` decides which
half the content comes from, and with no `.workspace` file it answers "this
repo" — so **steps 4 and 6 are the only ones that care.**

Two more things are missing from any fresh runtime, being the two `.gitignore`
keeps out of git entirely:

| | | |
|---|---|---|
| `engine/` | the upstream [ACE-Step 1.5](https://github.com/ace-step/ACE-Step-1.5) clone | ~5 GB of wheels |
| `engine/checkpoints/` | the weights | ~8 GB |

Step 5 fetches both. It costs about ten minutes and is paid again every time
Colab hands you a new machine, which is why **step 2 mounts Drive: so the songs
survive that.**

> **Runtime → Change runtime type → GPU** before you start. A free T4 renders a
> three-minute take in well under a minute, against roughly nine on a MacBook.

## 1 · What are we running on

In [ ]:
#@title Check the GPU { display-mode: "form" }
# The machine gets the last word on two settings the songs know nothing about:
# which device to load on, and which backend the 5Hz LM should use.
#
# nano-vllm (the `vllm` backend, which aimc picks by default on CUDA) wants
# bfloat16, and that arrived with Ampere — compute capability 8.0. On an older
# card, a T4 at 7.5, the engine would try it, fail, and fall back to PyTorch
# anyway; saying `pt` here skips a slow failure rather than changing the result.
import torch

if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> GPU, then re-run.")

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3

EXTRA = "--device cuda"
if (major, minor) < (8, 0):
    EXTRA += " --backend pt"

print(f"{name}  —  {vram:.0f} GiB, compute capability {major}.{minor}")
print(f"every take will be given: {EXTRA}")

## 2 · Somewhere for the songs to land

Colab throws the machine away, and a long queue outlasts a free session.
Mounting Drive is what makes a run survivable: step 7 copies each finished take
across as it lands, so a disconnect costs the take in flight and nothing else.

Untick it to render into the runtime and download a zip at the end instead.

In [ ]:
#@title Mount Google Drive { display-mode: "form" }
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_FOLDER = "AI_Music_Creator"  #@param {type:"string"}

from pathlib import Path

DRIVE = None
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    DRIVE.mkdir(parents=True, exist_ok=True)
    print(f"takes and ledger will be copied to: {DRIVE}")
else:
    print("no Drive: everything stays on this runtime and dies with it.")

## 3 · The tool

A few megabytes. `engine/` and the weights are not in here — step 5 fetches
those.

In [ ]:
#@title Clone or update the tool { display-mode: "form" }
REPO_URL = "https://github.com/YOUR-USERNAME/ai-music-creator.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}

import os
import subprocess
from pathlib import Path


def clone_or_pull(url: str, into: Path, branch: str) -> None:
    """Fetch, or fast-forward what is already here.

    --ff-only, and it is not timidity: a merge conflict resolved inside a
    machine that is about to be deleted is work thrown away. Edits happen on
    the Mac; this end only ever catches up to them.
    """
    if (into / ".git").exists():
        subprocess.run(["git", "-C", str(into), "fetch", "origin", branch], check=True)
        subprocess.run(["git", "-C", str(into), "checkout", branch], check=True)
        subprocess.run(["git", "-C", str(into), "merge", "--ff-only",
                        f"origin/{branch}"], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", branch, url, str(into)], check=True)


ROOT = Path("/content") / REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
clone_or_pull(REPO_URL, ROOT, BRANCH)

os.chdir(ROOT)
# uv installs to ~/.local/bin, and every wrapper in this repo runs through it.
os.environ["PATH"] = f"{Path.home()}/.local/bin:{os.environ['PATH']}"

print(subprocess.run(["git", "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout)
print(f"working directory: {Path.cwd()}")

## 4 · The content

**Skip this cell** to use the examples that came with the tool, and write your
own presets in `presets/`.

To render your own batch, this clones the private repo into `private/` and
writes `.workspace` — the one line that tells `aimc/workspace.py` to look there
instead of at the repo root. Nothing else in the tool changes, and none of your
presets, lyrics or job files need editing: they keep saying exactly what they
said on the Mac.

The token comes from **Colab Secrets** (🔑 in the left sidebar): add one named
`GITHUB_TOKEN`, holding a fine-grained personal access token with read access to
that one repository, and switch on *Notebook access*. It is never printed and
never written to disk.

In [ ]:
#@title Clone the private content repo { display-mode: "form" }
PRIVATE_REPO_URL = "https://github.com/YOUR-USERNAME/ai-music-private.git"  #@param {type:"string"}
PRIVATE_BRANCH = "main"  #@param {type:"string"}
USE_PRIVATE = True  #@param {type:"boolean"}

import subprocess
import sys
from pathlib import Path

if USE_PRIVATE:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
    # The token goes into the URL git is handed and into nothing else: not into
    # the remote it stores, not into a printed line, not onto disk.
    authed = PRIVATE_REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    clone_or_pull(authed, ROOT / "private", PRIVATE_BRANCH)
    subprocess.run(["git", "-C", "private", "remote", "set-url", "origin",
                    PRIVATE_REPO_URL], check=True)
    (ROOT / ".workspace").write_text("private\n")
else:
    (ROOT / ".workspace").unlink(missing_ok=True)

# aimc.workspace imports nothing but os and pathlib, so it answers this question
# long before the engine exists. Asking it, rather than repeating its rules
# here, is what keeps the notebook honest about where the content actually is.
sys.path.insert(0, str(ROOT))
for stale in [m for m in sys.modules if m.startswith("aimc")]:
    del sys.modules[stale]
from aimc.workspace import NIGHT, PRESETS, SONGS, WORKSPACE

print(f"\nworkspace : {WORKSPACE}")
n_presets = len(list(PRESETS.rglob("*.json"))) if PRESETS.exists() else 0
print(f"presets   : {PRESETS}  ({n_presets} files)")
print(f"takes     : {SONGS}")
print(f"queue     : {NIGHT / 'queue'}")


def to_drive() -> None:
    """Copy what has landed out of the machine that is going to be deleted.

    One direction only, workspace -> Drive. `.run-*` is the engine's staging
    folder for a take in flight and `.started-*` the runner's marker: copying
    either would copy a half-written wav and call it a song.
    """
    if DRIVE is None:
        return
    if SONGS.exists():
        (DRIVE / "songs").mkdir(parents=True, exist_ok=True)
        subprocess.run(["rsync", "-a", "--exclude", ".run-*", "--exclude",
                        ".started-*", f"{SONGS}/", str(DRIVE / "songs")], check=False)
    for f in (NIGHT / "ledger.tsv", ROOT / "night" / "worker-console.log"):
        if f.exists():
            subprocess.run(["cp", str(f), str(DRIVE / f.name)], check=False)

## 5 · The engine and the weights

Ten minutes, once per machine. [`colab/setup.sh`](colab/setup.sh) does what the
README's install section does on a Mac — clone ACE-Step, `uv sync`, download the
checkpoints — and it is idempotent, so re-running it after a reconnect only
checks.

`FULL_CHECKPOINTS` adds the 1.7B LM (3.5 GB). Leave it off unless a preset of
yours names it: `aimc/generation/catalog.py` defaults to the 0.6B.

In [ ]:
#@title Install the engine and download the weights { display-mode: "form" }
FULL_CHECKPOINTS = False  #@param {type:"boolean"}

!bash colab/setup.sh {"--full" if FULL_CHECKPOINTS else ""}

## 6 · What is in the queue

A job file is one take: a preset, a seed, a step count. The runner moves it to
`done/` or `failed/` and appends a line to `ledger.tsv`. That is the whole
protocol, and it is why an interrupted run resumes by starting again — whatever
is still in `queue/` is whatever is still owed.

Empty queue? On the Mac, `python3 night/build.py <collection>` fills it from a
collection module.

In [ ]:
#@title Queue status { display-mode: "form" }
import json

waiting = sorted((NIGHT / "queue").glob("*.json")) if (NIGHT / "queue").exists() else []


def count(folder: str) -> int:
    d = NIGHT / folder
    return len(list(d.glob("*.json"))) if d.exists() else 0


print(f"{len(waiting):>4} waiting   {count('done'):>4} done   "
      f"{count('failed'):>4} failed   {count('held'):>4} held\n")

for job_file in waiting[:10]:
    job = json.loads(job_file.read_text())
    print(f"  {job['collection']:<18} {job['slug']:<28} seed {job['seed']:<4} "
          f"{job['steps']} steps")
if len(waiting) > 10:
    print(f"  … and {len(waiting) - 10} more")

## 7 · Render the batch

This starts [`night/batch_render.py`](night/batch_render.py) — the worker that
holds the DiT and the 5Hz LM in memory and walks the queue, instead of paying
three and a half minutes of model loading per take.

**On the Mac that worker was built, measured and reverted.** 16 GiB could not
hold a DiT twice; a take hung for fifty minutes at `[DCW] Built DWT1D` without
ever failing, and `night/runner.sh` — one fresh process per take — is what runs
there. Here the ceiling that decided it is gone, and loading once is the entire
reason to rent a GPU.

It runs detached, so **interrupting this cell does not stop the render**, it
only stops watching. Re-run to pick the watch back up; step 8 stops it.

In [ ]:
#@title Start the worker and watch it { display-mode: "form" }
import os
import subprocess
import time

PIDFILE = ROOT / "night" / ".colab-worker.pid"
CONSOLE = ROOT / "night" / "worker-console.log"   # the file ledger.tsv names
LEDGER = NIGHT / "ledger.tsv"


def worker_pid() -> int | None:
    """The live worker, or None.

    A pid file rather than `pgrep -f batch_render.py`, because `worker.sh` execs
    into `uv run`, which then runs python: the pattern matches two processes and
    signalling the wrong one leaves the other rendering. The pid we wrote is the
    session leader of both, which is the handle we actually want.
    """
    if not PIDFILE.exists():
        return None
    pid = int(PIDFILE.read_text().strip())
    try:
        os.kill(pid, 0)          # signal 0: asks, does not touch
    except OSError:
        return None
    return pid


if worker_pid() is None:
    with CONSOLE.open("ab") as fh:
        # start_new_session: its own process group, so it outlives an
        # interrupted cell — and so step 8 can stop the whole group at once.
        proc = subprocess.Popen(["./night/worker.sh", "--extra", EXTRA],
                                stdout=fh, stderr=fh, start_new_session=True)
    PIDFILE.write_text(str(proc.pid))
    time.sleep(5)
    print(f"worker started (pid {proc.pid}) with: {EXTRA}\n")
else:
    print(f"worker already running (pid {worker_pid()}) — watching it\n")

seen = len(LEDGER.read_text().splitlines()) if LEDGER.exists() else 0
try:
    while True:
        rows = LEDGER.read_text().splitlines() if LEDGER.exists() else []
        for row in rows[seen:]:
            if row.startswith("finished_at"):
                continue
            _when, slug, coll, status, secs, audio, *_ = row.split("\t")
            print(f"  {'ok' if status == 'ok' else '✗ '} {coll}/{slug:<30} "
                  f"{secs:>4}s  {audio}")
        seen = len(rows)

        to_drive()

        if worker_pid() is None:
            # Before the first take lands there is nothing in the ledger and
            # nothing to see but the models loading; after a crash the same is
            # true and the reason is in the console. Show it either way.
            print("\n".join(CONSOLE.read_text(errors="replace").splitlines()[-12:]))
            left = len(list((NIGHT / "queue").glob("*.json")))
            print(f"\nworker stopped — {left} job(s) still queued")
            break
        time.sleep(30)
except KeyboardInterrupt:
    print(f"\nstopped watching. The worker (pid {worker_pid()}) is still "
          f"rendering — re-run this cell to watch it again.")

## 8 · Stop the worker

In [ ]:
#@title Stop after the take in flight { display-mode: "form" }
# SIGTERM to the whole process group: worker.sh, uv and python are all in it,
# and killing only the first would leave the third rendering. The take in flight
# is lost either way — but its job file is still in queue/, because a job only
# moves once its wav exists, so the next run picks it up untouched.
import os
import signal

if PIDFILE.exists():
    pid = int(PIDFILE.read_text().strip())
    try:
        os.killpg(os.getpgid(pid), signal.SIGTERM)
        print(f"sent SIGTERM to the worker's process group ({pid})")
    except OSError as exc:
        print(f"nothing to stop: {exc}")
    PIDFILE.unlink()
else:
    print("no worker running")

## 9 · A single take

The same `./song` the README documents, unchanged. Useful for trying a preset at
GPU speed before committing a hundred of them to the queue — a take that costs
nine minutes on the Mac costs seconds here, which is a different way of working
with the same tool.

Leave `PRESET` empty to use `STYLE` and `LYRICS_FILE` instead. Paths are
relative to the working directory, so a preset of your own lives under
`private/presets/…`.

In [ ]:
#@title Render one song { display-mode: "form" }
PRESET = ""  #@param {type:"string"}
STYLE = "Dub techno, 1993 Berlin, one filtered chord stab drenched in tape delay, muffled four to the floor kick, deep sub bass, enormous reverb, instrumental"  #@param {type:"string"}
LYRICS_FILE = ""  #@param {type:"string"}
SEED = 1  #@param {type:"integer"}
STEPS = 8  #@param {type:"integer"}

import shlex
import subprocess

argv = ["./song", "--seed", str(SEED), "--steps", str(STEPS),
        "--out", str(SONGS / "colab")]
if PRESET:
    argv += ["--preset", PRESET]
if STYLE and not PRESET:
    argv += ["--style", STYLE, "--instrumental"]
if LYRICS_FILE:
    argv += ["--lyrics", LYRICS_FILE]
argv += shlex.split(EXTRA)

print(" ".join(shlex.quote(a) for a in argv), "\n")
subprocess.run(argv, check=False)

In [ ]:
#@title Listen to the newest take { display-mode: "form" }
from IPython.display import Audio, display

takes = sorted(SONGS.rglob("*.wav"), key=lambda p: p.stat().st_mtime)
if not takes:
    print("nothing rendered yet")
else:
    newest = takes[-1]
    print(f"{newest.relative_to(WORKSPACE)}  ({newest.stat().st_size / 1024**2:.1f} MB)")
    display(Audio(str(newest)))

## 10 · Bring it home

With Drive mounted, step 7 has been copying as it went and this only catches the
last take. Without Drive, this is the one chance to get the audio off the
machine before it is reclaimed.

The takes do not come back through git — `songs/` is gitignored in both repos,
deliberately. What is worth committing afterwards is `night/ledger.tsv` and the
job files that moved into `night/done/`: the record of what was rendered, and
the only part of a run that re-running cannot reproduce.

In [ ]:
#@title Sync to Drive, or build a zip { display-mode: "form" }
import subprocess

if DRIVE is not None:
    to_drive()
    takes = list((DRIVE / "songs").rglob("*.wav"))
    print(f"{DRIVE}  —  {len(takes)} takes, "
          f"{sum(f.stat().st_size for f in takes) / 1024**3:.2f} GB")
else:
    archive = "/content/takes.zip"
    subprocess.run(["zip", "-r", "-q", archive, str(SONGS),
                    str(NIGHT / "ledger.tsv"), str(NIGHT / "done")], check=False)
    from google.colab import files

    files.download(archive)